In [1]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('pricing_diff.csv')
df.shape


(20000, 8)

In [5]:
df.head()

,order_id,weight_kg,distance_km,category,express,coupon,v1_total,v2_total
0,500000,22.48,280.0,books,False,NaN,49.97,54.47
1,500001,19.50,47.5,standard,False,NaN,41.38,41.36
2,500002,6.02,200.9,fragile,False,NaN,25.70,25.68
3,500003,7.85,25.3,electronics,False,SAVE10,22.33,22.31
4,500004,21.90,245.5,clothing,False,NaN,56.08,56.07


In [6]:
df.dtypes

order_id         int64
weight_kg      float64
distance_km    float64
category           str
express           bool
coupon             str
v1_total       float64
v2_total       float64
dtype: object

In [15]:
df['diff']= df.v2_total - df.v1_total


In [16]:
df.head()

,order_id,weight_kg,distance_km,category,express,coupon,v1_total,v2_total,diff
0,500000,22.48,280.0,books,False,NaN,49.97,54.47,4.50
1,500001,19.50,47.5,standard,False,NaN,41.38,41.36,-0.02
2,500002,6.02,200.9,fragile,False,NaN,25.70,25.68,-0.02
3,500003,7.85,25.3,electronics,False,SAVE10,22.33,22.31,-0.02
4,500004,21.90,245.5,clothing,False,NaN,56.08,56.07,-0.01


In [17]:
# Q1 Answer
count_zeros = (df['diff'] == 0).sum()
print(count_zeros)


4000


In [18]:
print(df['diff'].describe())

count    20000.000000
mean         1.401492
std          4.790864
min         -0.020000
25%         -0.010000
50%          0.010000
75%          0.020000
max         34.980000
Name: diff, dtype: float64



75% of the data is below 0.02 but the max is 34.98 which indicates a very huge jump which mean that the bug is not a float noise or order size,
mean is 1.4 and median is 0.01 which indicates that there is a small sample of data which drags the average around, so we group by category to SEEK_CURwhich rows are causing the issue


In [21]:
print(df.groupby('category')['diff'].describe())

              count      mean        std   min   25%   50%   75%    max
category                                                               
books        3335.0  2.475028   1.408762  0.08  1.25  2.40  3.71   5.01
clothing     3277.0 -0.000018   0.012130 -0.02 -0.01  0.00  0.01   0.02
electronics  3353.0 -0.000140   0.012076 -0.02 -0.01  0.00  0.01   0.02
food         3371.0 -0.000374   0.012099 -0.02 -0.01  0.00  0.01   0.02
fragile      3325.0  5.948692  10.313943 -0.02 -0.01  0.01  9.74  34.98
standard     3339.0 -0.000596   0.012238 -0.02 -0.01  0.00  0.01   0.02


all cat except fragile is clean as they are uniform, in fragile some values are near 0 and some very high and nothing in between so we inspect that

In [22]:
frag = df[df.category == 'fragile']
print(frag.groupby(['express', frag.coupon.fillna('NONE')])['diff'].describe())

                 count       mean       std   min    25%    50%    75%    max
express coupon                                                               
False   NONE    1874.0   0.000181  0.012212 -0.02  -0.01   0.00   0.01   0.02
        SAVE10   466.0  -0.000172  0.012056 -0.02  -0.01   0.00   0.01   0.02
True    NONE     772.0  20.654132  8.812411  5.66  12.81  21.11  28.27  34.98
        SAVE10   213.0  18.000704  7.842092  5.08  11.05  17.82  24.72  31.22


when express is false, every parameter is in range, but in express true the values are up

In [23]:
affected = df[(df['category'] == 'fragile') & (df['express'])]
print(affected['diff'].corr(affected['weight_kg']))
print(affected['diff'].corr(affected['distance_km']))

0.01809690793232644
0.9945622741566471


correlation with weight is 0.02 i.e nothing
but with distance it is 0.99 near perfect i.e linear relation (ax + b)

In [26]:

total_overcharged = affected['diff'].sum()
print(f"Total amount overcharged: {total_overcharged}")

Total amount overcharged: 19779.14


In [31]:
non_book_order = df['category'] != 'books'
unaffected = ~((df['category'] == 'fragile') & (df['express'] == True))

average_diff = df[unaffected & non_book_order]['diff'].mean()
print(f"Average diff for unaffected, non-book orders: {average_diff}")

Average diff for unaffected, non-book orders: -0.00022448979591839984


Average diff for unaffected, non-book orders: 1.401492
